In [117]:
# import necessary libraries

import pandas as pd
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [118]:
# Import Dataset

data = pd.read_csv("Reviews.csv")

In [119]:
data

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...,...,...,...,...,...,...,...
568449,568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if no...
568450,568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,I'm disappointed with the flavor. The chocolat...
568451,568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568452,568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rew...


In [120]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Id                      568454 non-null  int64 
 1   ProductId               568454 non-null  object
 2   UserId                  568454 non-null  object
 3   ProfileName             568428 non-null  object
 4   HelpfulnessNumerator    568454 non-null  int64 
 5   HelpfulnessDenominator  568454 non-null  int64 
 6   Score                   568454 non-null  int64 
 7   Time                    568454 non-null  int64 
 8   Summary                 568427 non-null  object
 9   Text                    568454 non-null  object
dtypes: int64(5), object(5)
memory usage: 43.4+ MB


In [121]:
data.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

In [122]:
data.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [123]:
df["Score"].value_counts()

Score
5    363122
4     80655
1     52268
3     42638
2     29744
Name: count, dtype: int64

In [124]:
# Data preprocessing

df = data[["Score", "Summary", "Text"]]

In [125]:
df

,Score,Summary,Text
0,5,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,4,"""Delight"" says it all",This is a confection that has been around a fe...
3,2,Cough Medicine,If you are looking for the secret ingredient i...
4,5,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...
568449,5,Will not do without,Great for sesame chicken..this is a good if no...
568450,2,disappointed,I'm disappointed with the flavor. The chocolat...
568451,5,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568452,5,Favorite Training and reward treat,These are the BEST treats for training and rew...


In [126]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Score    568454 non-null  int64 
 1   Summary  568427 non-null  object
 2   Text     568454 non-null  object
dtypes: int64(1), object(2)
memory usage: 13.0+ MB


In [127]:
df.isnull().sum()

Score       0
Summary    27
Text        0
dtype: int64

In [128]:
# Drop missing values

df.dropna(inplace=True)

C:\Users\Kaushal\AppData\Local\Temp\ipykernel_33404\1755519314.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [129]:
df.isnull().sum()

Score      0
Summary    0
Text       0
dtype: int64

In [130]:
# convert rating score into sentiment

def sentiment(score):
    if score <= 2:
        return "Negative"
    elif score == 3:
        return "Neutral"
    else:
        return "Positive"

df["sentiment"] = df["Score"].apply(sentiment)

C:\Users\Kaushal\AppData\Local\Temp\ipykernel_33404\1245898913.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["sentiment"] = df["Score"].apply(sentiment)


In [131]:
df

,Score,Summary,Text,sentiment
0,5,Good Quality Dog Food,I have bought several of the Vitality canned d...,Positive
1,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,Negative
2,4,"""Delight"" says it all",This is a confection that has been around a fe...,Positive
3,2,Cough Medicine,If you are looking for the secret ingredient i...,Negative
4,5,Great taffy,Great taffy at a great price. There was a wid...,Positive
...,...,...,...,...
568449,5,Will not do without,Great for sesame chicken..this is a good if no...,Positive
568450,2,disappointed,I'm disappointed with the flavor. The chocolat...,Negative
568451,5,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o...",Positive
568452,5,Favorite Training and reward treat,These are the BEST treats for training and rew...,Positive


In [132]:
# Combine summary and text in review column

df["review"] = df["Summary"] + " " + df["Text"]


C:\Users\Kaushal\AppData\Local\Temp\ipykernel_33404\1086856123.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["review"] = df["Summary"] + " " + df["Text"]


In [133]:
# Drop extra same info columns

df.drop(columns=["Summary", "Text"], inplace=True)

C:\Users\Kaushal\AppData\Local\Temp\ipykernel_33404\667314044.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=["Summary", "Text"], inplace=True)


In [134]:
df

,Score,sentiment,review
0,5,Positive,Good Quality Dog Food I have bought several of...
1,1,Negative,Not as Advertised Product arrived labeled as J...
2,4,Positive,"""Delight"" says it all This is a confection tha..."
3,2,Negative,Cough Medicine If you are looking for the secr...
4,5,Positive,Great taffy Great taffy at a great price. The...
...,...,...,...
568449,5,Positive,Will not do without Great for sesame chicken.....
568450,2,Negative,disappointed I'm disappointed with the flavor....
568451,5,Positive,Perfect for our maltipoo These stars are small...
568452,5,Positive,Favorite Training and reward treat These are t...


In [135]:
stop_words = set(stopwords.words("english"))

def cleaned_text(text):
    # Tokenize the text
    words = nltk.word_tokenize(text.lower())

    # Remove stop words and non-alpabetic characters
    cleaned = [word for word in words if word.isalpha() and word not in stop_words]

    return ' '.join(cleaned)

# Apply preprocessing on review column
df["review"] = df["review"].apply(cleaned_text)

C:\Users\Kaushal\AppData\Local\Temp\ipykernel_33404\330200528.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["review"] = df["review"].apply(cleaned_text)


In [136]:
df

,Score,sentiment,review
0,5,Positive,good quality dog food bought several vitality ...
1,1,Negative,advertised product arrived labeled jumbo salte...
2,4,Positive,delight says confection around centuries light...
3,2,Negative,cough medicine looking secret ingredient robit...
4,5,Positive,great taffy great taffy great price wide assor...
...,...,...,...
568449,5,Positive,without great sesame chicken good better restu...
568450,2,Negative,disappointed disappointed flavor chocolate not...
568451,5,Positive,perfect maltipoo stars small give one training...
568452,5,Positive,favorite training reward treat best treats tra...


In [137]:
df['review'][0]

'good quality dog food bought several vitality canned dog food products found good quality product looks like stew processed meat smells better labrador finicky appreciates product better'

In [138]:
# Convert text into numbers and split columns into feature and target variable 

vectorizer = TfidfVectorizer()

x = vectorizer.fit_transform(df["review"])

y = df["sentiment"]

In [139]:
# splitting data into training and testing datset

x_train, x_test, y_train, y_test = train_test_split(x,y, train_size= 0.8, random_state=42)

In [140]:
# Training the model

model = LogisticRegression()

model.fit(x_train, y_train)

y_pred = model.predict(x_test)


C:\Users\Kaushal\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [141]:
# Evalute the model using metrice

acc_score = accuracy_score(y_test, y_pred)
print(f"Accuracy Score =",{acc_score})

Accuracy Score = {0.8868374294108333}


In [143]:
review = "I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most."

clean_review = cleaned_text(review)

In [144]:
trans_clean = vectorizer.transform([clean_review])

In [145]:
model.predict(trans_clean)

array(['Positive'], dtype=object)

In [146]:
# Save model along with vectorizer for making streamlit interface app


import joblib

model_path = 'Amazone_review_Model.joblib'
joblib.dump(model, model_path)

vect_path = 'Vectorizer.joblib'
joblib.dump(vectorizer, vect_path)

['Vectorizer.joblib']